In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.colors import LogNorm
#load our data

Secondary_Filter_Deleterious_bulk = pd.read_csv(r"SLIMs/Data/Results_Bulk_Secondary_Filter_Deleterious.csv",header=0)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

# Define the plot colors.
color_1 = "#2D918EEA"  # Teal (Highly significant hue)

# Low log values (insignificant) are white, hitting white at the center benchmark,
# and scaling smoothly to teal for high log values (significant).
custom_diverging_map = LinearSegmentedColormap.from_list(
    "custom_sharp_transition", 
    ["#FFFFFF", "#FFFFFF", color_1] # Faded/White -> White Midpoint -> Teal Hue
)

def barplot_top_n_hits(dataframe, n, title):
    # Ensure dataframe is sorted by q-value ascending (most significant at top)
    sorted_df = dataframe.sort_values(by="q-value", ascending=True)
    top_n_df = sorted_df.head(n).copy()

    fig, axs = plt.subplots(figsize=(10, 10))

    # Convert q-values to -log10 space safely
    q_values_safe = top_n_df["q-value"].clip(lower=1e-300)
    log_q = -np.log10(q_values_safe)

    vmax_log = log_q.max()       
    vmin_log = log_q.min()       
    vcenter_log = -np.log10(0.1) # Center benchmark threshold locked at FDR 0.1

    # Prevent normalization scale collapse if data variance is narrow
    if vmax_log <= vcenter_log:
        vmax_log = vcenter_log + 1
    if vmin_log >= vcenter_log:
        vmin_log = vcenter_log - 1

    # Maps vmin_log to white, vcenter_log to white, and vmax_log to teal
    norm = TwoSlopeNorm(vmin=vmin_log, vcenter=vcenter_log, vmax=vmax_log)
    sm = cm.ScalarMappable(cmap=custom_diverging_map, norm=norm)

    # Underlying chart framework with a thin dark border around bars
    barplot = sns.barplot(
        data=top_n_df, 
        x="Odds Ratio", 
        y="SLIM Motif", 
        color="gray", 
        ax=axs,
        edgecolor="#4A4A4A", 
        linewidth=0.8
    )

    # Assign calculated color parameters to individual data rows
    for bar, l_q in zip(barplot.patches, log_q):
        color = sm.to_rgba(l_q)
        bar.set_facecolor(color)

    # =========================================================================
    # DYNAMIC SIGNIFICANCE DIVIDER LINE
    # =========================================================================
    # Find the position where q-value crosses above 0.1
    # Since the dataframe is ordered, look for the first index exceeding 0.1
    q_vals_list = top_n_df["q-value"].tolist()
    divider_y_idx = None
    
    for idx, q_val in enumerate(q_vals_list):
        if q_val > 0.1:
            divider_y_idx = idx
            break

    # If a crossing point exists inside this slice, draw the line between the bars
    if divider_y_idx is not None and divider_y_idx > 0:
        # In a horizontal Seaborn barplot, the bar gaps sit exactly at half-integers
        line_position = divider_y_idx - 0.5
        
        axs.axhline(
            y=line_position, 
            color="#D05570",       # Noticeable contrasting color (or choose 'black')
            linestyle="--", 
            linewidth=1.2, 
            zorder=3,              # Places the line on top of the bars
            label="FDR = 0.1 Threshold"
        )
        # Optional: Adds a small text marker just above the line path
        axs.text(
            x=axs.get_xlim()[1] * 0.98, 
            y=line_position - 0.1, 
            s="Significant (FDR < 0.1)", 
            color="#4A4A4A", 
            fontsize=9, 
            ha="right", 
            va="bottom"
        )
    # =========================================================================

    # Colorbar canvas assembly
    sm.set_array([])
    cbar = axs.figure.colorbar(sm, ax=axs, label="FDR-Adjusted P-Value (q-value)")

    # Generate full floor/ceil range of integer log exponents spanning your data
    min_exponent = int(np.floor(vmin_log))
    max_exponent = int(np.ceil(vmax_log))
    
    total_orders_of_magnitude = max_exponent - min_exponent
    step = 2 if total_orders_of_magnitude > 8 else 1

    # Generate clean, standard tick positions matching powers of 10
    log_ticks = np.arange(1, max_exponent + 1, step)
    if min_exponent < 1:
        lower_log_ticks = np.arange(1 - step, min_exponent - 1, -step)
        log_ticks = np.concatenate((lower_log_ticks[::-1], log_ticks))

    log_ticks = log_ticks[(log_ticks >= vmin_log) & (log_ticks <= vmax_log)]

    if len(log_ticks) == 0:
        log_ticks = np.array([vcenter_log])

    fixed_ticks = 10.0 ** (-log_ticks)

    cbar.set_ticks(log_ticks)
    cbar.set_ticklabels([f"{x:.1g}" if x >= 0.01 else f"{x:.0e}" for x in fixed_ticks])
    cbar.ax.invert_yaxis()

    plt.title(title)
    return fig


In [ ]:
n=25
for dataframe in [(Secondary_Filter_Deleterious_bulk, "Enrichment in Deleterious Peptides, Bulk Analysis")]:
    barplot_top_n_hits(dataframe[0], n, dataframe[1])
    plt.savefig(f"SLIMs/Results/Plots/Supplement_2_F_{dataframe[1]}.pdf", dpi=600)
    plt.show()
    
    
""" 
PLEASE NOTE THAT SUPPLEMENT 2 F WAS EDITED IN ADOBE ILLUSTRATOR FOR CUSTOM APPEARANCE. SHOWN BELOW IS THE EXACT PLOT USED FOR THAT.

"""
    
